# 06 — Improvements (validation only)
Her deneme **tek bir değişiklik** içerir ve sadece **validation** setinde karşılaştırılır (test seti en sona saklanıyor). Biten modeller tekrar eğitilmez; yarıda kalan eğitim kaldığı epoch'tan devam eder (bilgisayar kapanırsa / Colab koparsa hücreyi tekrar çalıştır).

In [ ]:
# On Colab: mount Drive and work from the project's notebooks folder
import os, sys
if "google.colab" in sys.modules and not os.getcwd().endswith("notebooks"):
    from google.colab import drive
    drive.mount("/content/drive")
    %cd /content/drive/MyDrive/bone-age/notebooks
%run 00_common.ipynb

In [ ]:
EXPERIMENTS = {
    "b0_512": dict(backbone="efficientnet_b0", img_size=512),  # 1. larger images
    "b3_384": dict(backbone="efficientnet_b3", img_size=384),  # 2. bigger backbone
}
for name, kwargs in EXPERIMENTS.items():
    print(f"=== {name} ===")
    train_model(name, **kwargs)  # skips finished runs, resumes interrupted ones

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name in ["baseline_b0_384", *EXPERIMENTS]:
    log = pd.read_csv(OUTPUT_DIR / name / "log.csv")
    ax.plot(log["epoch"], log["val_mae_months"], "o-", label=name)
ax.set(xlabel="epoch", ylabel="val MAE (months)", ylim=(None, 12), title="Validation MAE per epoch")
ax.legend()
fig.savefig(OUTPUT_DIR / "experiments_val_curves.png", dpi=120, bbox_inches="tight")
plt.show()

3. TTA ve 4. ensemble (birkaç modelin ortalaması) — eğitim gerektirmez, sadece tahmin:

In [ ]:
candidates = [
    (["baseline_b0_384"], False),
    (["b0_512"], False),
    (["b3_384"], False),
    (["baseline_b0_384"], True),
    (["b0_512"], True),
    (["b3_384"], True),
    (["baseline_b0_384", "b0_512", "b3_384"], False),
    (["baseline_b0_384", "b0_512", "b3_384"], True),
]
rows = []
for runs, tta in candidates:
    _, m = evaluate(runs, "val", tta)
    rows.append({"setup": " + ".join(runs) + (" + TTA" if tta else ""), "runs": runs, "tta": tta,
                 "val_mae": m["mae"], "val_rmse": m["rmse"], "val_r2": m["r2"]})
results = pd.DataFrame(rows).sort_values("val_mae").reset_index(drop=True)
results.to_json(OUTPUT_DIR / "experiments.json", orient="records", indent=2)
results[["setup", "val_mae", "val_rmse", "val_r2"]].round(3)